# 0. Engine checks — SM Table SII

Two checks that everything downstream rests on, plus the symbolic checks of the vertex algebra.

**(a) Against an independent Schrödinger-picture recursion.** The coefficients of $\langle a^\dagger a\rangle=\sum_N c_N U^N$ for the driven Kerr cavity are recomputed as $\rho^{(N)}=-\mathcal L_0^{+}\mathcal L_{\rm int}\rho^{(N-1)}$ in a truncated Fock basis: different picture, different basis, no labels, no vertex table, no displaced frame.

**(b) The inverse Wick map** (Sec. IV F), which turns eigenoperator moments into physical ones, checked on a squeezed-thermal steady state (`engines/check_inversion.py`), and **(c)** the symbolic derivation of the four elementary moves and of the emitter moves (`engines/check_wick.py`, sympy).

**Conventions.** Schrödinger-picture generators use $\mathcal L(\rho)=-i[H,\rho]+\mathcal D(\rho)$; adjoint (Heisenberg) generators use $\mathcal L^\dagger(O)=+i[H,O]+\mathcal D^\dagger(O)$. Both appear in `engines/`; the plus sign is correct only for the adjoint. Every tilted (counting-field) calculation is checked against $c_1=\kappa\langle a^\dagger a\rangle$, which is violated if that sign is wrong.

The full list of checks is Table SII of the Supplemental Material; the multimode-vs-single-mode and tensor-vs-momentum comparisons are in notebook 06, the K = 3 counting check in notebook 08.

In [1]:
from common import *
FAST = True      # True: reduced orders and cutoffs, minutes.  False: the orders and cutoffs of the paper (see the runtime note)
print("repository:", os.path.basename(CODE), "| FAST =", FAST, "| paper sources:", "found" if PAPER else "not present (Tables V and SI from notebooks/paper_reference/)")

repository: gh_work | FAST = True | paper sources: not present (Tables V and SI from notebooks/paper_reference/)


## (a) Diagram coefficients versus the Schrödinger-picture recursion

In [2]:
import scipy.sparse as sps, scipy.sparse.linalg as spl
kappa, Dc, eta = 1.0, -1.0, 1.0
z = kappa/2 + 1j*Dc; alpha = eta/z
a, ad, b, bd = mode_ops(alpha)
mod = Model(1, z, lambda X: 0*X, np.eye(1), scale(0.5, mul(mul(ad,ad), mul(a,a))), 4)     # Kerr vertex, U = 1
Nchk = 8 if FAST else 16
c_diag = moment_series(mod, 1, 1, alpha, Nchk).real

Nc = 30 if FAST else 40
A = destroy(Nc); Ad = A.conj().T
H0 = Dc*Ad@A + 1j*(eta*Ad - np.conj(eta)*A)
L0 = liouvillian(H0, [np.sqrt(kappa)*A]); L1 = liouvillian(0.5*Ad@Ad@A@A, [])
rho0 = steady(H0, [np.sqrt(kappa)*A]).reshape(-1, order='F')
tr = np.eye(Nc).reshape(-1, order='F')
M = np.vstack([np.hstack([L0, tr[:,None]]), np.hstack([tr[None,:], [[0]]])])
lu = spl.splu(sps.csc_matrix(M))
c_rec = [np.trace(Ad@A@rho0.reshape(Nc,Nc,order='F')).real]; r = rho0
for N in range(1, Nchk+1):
    sol = lu.solve(np.concatenate([-(L1@r), [0]])); r = sol[:-1]
    c_rec.append(np.trace(Ad@A@r.reshape(Nc,Nc,order='F')).real)
print(" N   diagrams          recursion         rel. diff")
for N in range(0, Nchk+1, max(1, Nchk//6)):
    print(f"{N:3d}  {c_diag[N]: .8e}  {c_rec[N]: .8e}  {abs(c_diag[N]/c_rec[N]-1):.1e}")
print(f"\nmax relative difference through N={Nchk}: {max(abs(c_diag[N]/c_rec[N]-1) for N in range(Nchk+1)):.1e}   (paper: 1e-14)")

 N   diagrams          recursion         rel. diff
  0   8.00000000e-01   8.00000000e-01  4.4e-16
  1   1.02400000e+00   1.02400000e+00  5.6e-15
  2   2.77504000e+00   2.77504000e+00  7.8e-15
  3   9.35526400e+00   9.35526400e+00  1.2e-14
  4   3.60292844e+01   3.60292844e+01  8.9e-15
  5   1.51877706e+02   1.51877706e+02  2.2e-15
  6   6.84371996e+02   6.84371996e+02  2.2e-14
  7   3.24749823e+03   3.24749823e+03  4.8e-14
  8   1.60587908e+04   1.60587908e+04  5.7e-14

max relative difference through N=8: 5.7e-14   (paper: 1e-14)


## (b) Wick inversion and (c) symbolic vertex checks

In [3]:
run("python3 check_inversion.py", "engines", tail=3)
run("python3 check_wick.py", "engines", tail=6)

$ (cd code/engines; python3 check_inversion.py)   [6.6 s, exit 0]
inversion residual: 2.2737367544323206e-13  and O_00 coefficient = steady moment: ok
examples: <a^dag a> -> {(1, 1): np.complex128(1+0j), (0, 0): np.complex128(0.30545266060365567-3.2232516571388634e-18j)} 
  (a^dag)^2a^2 -> {(2, 2): np.complex128(1+0j), (1, 1): np.complex128(1.2218-0j), (0, 0): np.complex128(0.2641-0j), (2, 0): np.complex128(-0.2355+0.1484j), (0, 2): np.complex128(-0.2355-0.1484j)}


$ (cd code/engines; python3 check_wick.py)   [3.4 s, exit 0]
general boson moves: True
s+ O {(0, 0): {(1, 0): 1}, (0, 1): {(0, 0): p_e, (1, 1): 1}, (1, 0): {}, (1, 1): {(1, 0): -p_e}}
O s- {(0, 0): {(0, 1): 1}, (0, 1): {}, (1, 0): {(0, 0): p_e, (1, 1): 1}, (1, 1): {(0, 1): -p_e}}
s- O {(0, 0): {(0, 1): 1}, (0, 1): {}, (1, 0): {(0, 0): 1 - p_e, (1, 1): -1}, (1, 1): {(0, 1): 1 - p_e}}
O s+ {(0, 0): {(1, 0): 1}, (0, 1): {(0, 0): 1 - p_e, (1, 1): -1}, (1, 0): {}, (1, 1): {(1, 0): 1 - p_e}}


## Summary of SM Table SII

| check | where | result |
|---|---|---|
| diagram coefficients vs Schrödinger-picture recursion | above | 1e-14 |
| Wick inversion on a squeezed-thermal steady state | `check_inversion.py`, above | 1e-13 |
| four elementary moves and emitter moves, symbolic | `check_wick.py`, above | exact |
| multimode engine vs single-mode engine at K = 1 | notebook 06 | exact |
| tensor engine vs momentum-space engine at K = 3 | notebook 06 | exact |
| dressed emitter, fourth order, vs exact steady state | notebook 05 | 0.4 % |
| counting cumulants vs exact tilted Liouvillian, K = 1 and 3 | notebook 08 | Table V |
| trajectory estimator vs exact long-time cumulants, K = 1 and 3 | notebook 09 | within 1.6 s.e. |